# RevProof sparse-editor experiment (Colab / Colab Pro)
Upload the current committed `revproof.zip` made by `work/package_revproof.py`. This notebook compares prompt-only and LoRA SFT on the **dev** split. The existing v2 test was inspected, so it is excluded from this architecture comparison. Record the actual GPU; Colab Pro does not guarantee a GPU model.


In [ ]:
import subprocess, torch
print(subprocess.check_output(['nvidia-smi'], text=True))
print('torch', torch.__version__, 'CUDA build', torch.version.cuda)
assert torch.cuda.is_available(), 'Choose a GPU runtime first'
print('GPU', torch.cuda.get_device_name(0), 'free/total bytes', torch.cuda.mem_get_info())


In [ ]:
from google.colab import files
from pathlib import Path
import os, zipfile
uploaded = files.upload()  # select the latest revproof.zip
archives = [name for name in uploaded if name.endswith('.zip')]
assert len(archives) == 1, 'Upload exactly one project zip'
with zipfile.ZipFile(archives[0]) as zf:
    for entry in zf.infolist():
        parts = Path(entry.filename).parts
        assert parts and parts[0] == 'revproof' and '..' not in parts, entry.filename
    zf.extractall('/content')
os.chdir('/content/revproof')
print('project', Path.cwd())
assert Path('run_sparse_pilot.py').exists(), 'Zip is old; regenerate it from the latest repository commit'


In [ ]:
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.6', 'peft==0.21.2', 'accelerate==1.15.0',
    'bitsandbytes==0.50.2', 'sentencepiece>=0.2,<1'], check=True)
import transformers, peft, accelerate, bitsandbytes
print('transformers', transformers.__version__, 'peft', peft.__version__,
      'accelerate', accelerate.__version__, 'bitsandbytes', bitsandbytes.__version__)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-p', 'test_*.py', '-v'], check=True)


## First experiment: FLAN-T5-large editor
This trains a 770M-class encoder-decoder with LoRA. Keep the complete log, predictions, protocol, and adapter. A successful run is a measurement, not proof of semantic validity.


In [ ]:
subprocess.run([sys.executable, 'run_sparse_pilot.py',
    '--backend', 'seq2seq', '--model', 'google/flan-t5-large',
    '--data', 'data/real_v2.jsonl', '--output', 'runs/sparse_flan_large',
    '--stage', 'all'], check=True)


## Second experiment: same Qwen backbone, sparse output
Run this after the FLAN experiment. It isolates the edit-script task change from the encoder-decoder backbone change. It needs a CUDA GPU compatible with 4-bit bitsandbytes.


In [ ]:
subprocess.run([sys.executable, 'run_sparse_pilot.py',
    '--backend', 'causal', '--model', 'Qwen/Qwen2.5-1.5B-Instruct',
    '--data', 'data/real_v2.jsonl', '--output', 'runs/sparse_qwen_1p5b',
    '--stage', 'all'], check=True)


## Save results before the Colab runtime ends
Download predictions, summaries, and final LoRA adapters. Intermediate trainer checkpoints are excluded to keep the archive smaller. Do not report literal checks as human factual validity.


In [ ]:
import zipfile
out = Path('/content/revproof_sparse_results.zip')
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root in (Path('runs/sparse_flan_large'), Path('runs/sparse_qwen_1p5b')):
        if root.exists():
            for path in root.rglob('*'):
                if path.is_file() and not any(x.startswith('checkpoint-') for x in path.parts):
                    zf.write(path, path)
print(out, out.stat().st_size, 'bytes')
files.download(str(out))
